# Depth vs. Width — iso-budget generation on MATH-500 (Qwen3-1.7B / Qwen3-4B)

**One session = one (model, thinking-mode, level) triple.** Set `MODEL_NAME`, `THINK`, `LEVEL` in the config cell, run all.
Each session uses **43 problems** of that level (43 = all of level 1; levels 2–5 are a seeded random subset, identical across models/modes)
and writes a single file `/kaggle/working/outputs/<model>/<think|nothink>/level<LEVEL>.jsonl`.

20 sessions total: 2 models × {think, nothink} × levels {1..5}. The think sessions dominate cost.

**Kaggle settings:** Accelerator = *GPU T4 x2* (tensor-parallel over both), Internet = *On* (HF downloads).

**Efficiency trick (statistically exact):** a sample capped at `L` tokens is exactly the first `L` tokens of the same sample capped at 8000.
So we generate `K=16` samples **once at 8000 tokens** per problem and derive every shorter length by *token-prefix truncation + budget forcing*
(`</think> The final answer is \boxed{` + 24 greedy tokens). One 8K generation run + one cheap forcing pass yields all five lengths, the iso-budget diagonal,
the full 5×5 (N, L) surface, and bootstrap CIs.

**Collecting results:** after each session, Save Version (or download `outputs/`). The summary cell at the end also accepts `EXTRA_INPUTS` — paths of previously uploaded output datasets — so you can read all finished files together.

In [1]:
# ============================ CONFIG ============================
MODEL_NAME   = "Qwen/Qwen3-4B"      # "Qwen/Qwen3-1.7B" | "Qwen/Qwen3-4B"
THINK        = True                    # True = thinking mode, False = no-think. Run both.
LEVEL        = 3                       # MATH-500 level for this session: 1..5
N_PER_LEVEL  = 43                      # problems per level (43 = size of level 1; other levels are seeded subsets)

K            = 16                      # samples per (problem, length)
LENGTHS      = [8000, 4000, 2000, 1000, 500]   # iso-budget grid: 1x8000, 2x4000, 4x2000, 8x1000, 16x500
FORCE_TOKENS = 24                      # budget-forcing continuation length
SEED         = 0

EXTRA_INPUTS = []                      # summary cell only: e.g. ["/kaggle/input/depth-width-l1/outputs"]
PUSH_EVERY   = 1                       # push outputs/ to GitHub every this many newly completed problems

# vLLM
GPU_MEM_UTIL  = 0.92
ENFORCE_EAGER = False                  # set True if you hit CUDA-graph OOM on T4
# ===============================================================
import os
MAX_L      = max(LENGTHS)
MODE       = "think" if THINK else "nothink"
MODEL_TAG  = MODEL_NAME.split("/")[-1]
assert LEVEL in (1, 2, 3, 4, 5)
# "Batch" = headless Save & Run All (survives closing the browser); "Interactive" = draft session.
print("run type:", os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "?"), "| first one is running...")
# OUT_ROOT / OUT_DIR / OUT_PATH are set in the GITHUB SYNC cell below, once the repo is cloned.

run type: Batch | first one is running...


In [2]:
# ============================ GITHUB SYNC ============================
# outputs/ lives inside a clone of this repo, so every checkpoint write already sits in a
# git working tree. push_outputs() commits + pushes after each completed problem, so a
# session that dies mid-run leaves its progress on GitHub. A fresh session re-clones,
# sees exactly what was already pushed, and the RUN cell resumes from there instead of
# regenerating finished problems.
import os, subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_REPO   = "bhuwanadhikari/sequence-vs-parallel-cot-in-slm"
GITHUB_BRANCH = "main"
REPO_DIR      = "/kaggle/working/repo"

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")   # Add-ons -> Secrets, add one named GITHUB_TOKEN
REMOTE_URL   = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, REMOTE_URL, REPO_DIR], check=True)
    subprocess.run(["git", "config", "user.email", "kaggle@kaggle.com"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "config", "user.name", "Kaggle"], cwd=REPO_DIR, check=True)

OUT_ROOT = f"{REPO_DIR}/outputs"
OUT_DIR  = f"{OUT_ROOT}/{MODEL_TAG}/{MODE}"
OUT_PATH = f"{OUT_DIR}/level{LEVEL}.jsonl"
os.makedirs(OUT_DIR, exist_ok=True)
print(MODEL_TAG, MODE, f"level {LEVEL}", "->", OUT_PATH)

def push_outputs(message):
    """Commit outputs/, rebase onto anything another session pushed since, push. No-op if nothing changed."""
    subprocess.run(["git", "add", "outputs"], cwd=REPO_DIR, check=True)
    if subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=REPO_DIR).returncode == 0:
        return  # nothing new to push
    subprocess.run(["git", "commit", "-m", message], cwd=REPO_DIR, check=True)
    # Commit BEFORE pulling: once a level file has been pushed it is tracked, and every later append
    # is an unstaged change that makes `git pull --rebase` refuse to run ("You have unstaged changes").
    subprocess.run(["git", "pull", "--rebase", "origin", GITHUB_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "push", "origin", GITHUB_BRANCH], cwd=REPO_DIR, check=True)

Cloning into '/kaggle/working/repo'...


Qwen3-4B think level 3 -> /kaggle/working/repo/outputs/Qwen3-4B/think/level3.jsonl


In [3]:
# ============================ INSTALL ============================
# vLLM brings its own torch; install BEFORE importing torch. ~5-8 min on Kaggle.
import subprocess, sys, importlib.metadata as md

if "vllm" in sys.modules:
    # Re-running this cell in a live kernel would purge+re-import modules that are already in use
    # (e.g. datasets re-registering its pyarrow extension types -> ArrowKeyError). Nothing to do.
    print("INSTALL already ran in this kernel -- skipping. Restart the kernel to reinstall.")
elif "torch" in sys.modules:
    # torch is a C extension: once the preinstalled build is in memory, pip can replace the files on
    # disk but not the loaded libraries. `import torch, vllm` below would then pair a fresh vllm with
    # the stale torch -> "libcudart.so.13: cannot open shared object file" / "undefined symbol".
    import torch
    raise RuntimeError(f"torch {torch.__version__} was imported before INSTALL ran. "
                       "Restart the session (Run -> Restart & clear cell outputs), then Run All.")
else:
    HF_PKGS = ("huggingface_hub", "transformers", "tokenizers", "datasets", "safetensors")
    preloaded = sorted(m for m in sys.modules if m.split(".")[0] in HF_PKGS)
    print("already imported before install:", [m for m in preloaded if "." not in m] or "none")

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "vllm", "math-verify", "datasets"], check=True)
    hf_ver = md.version("huggingface_hub")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", f"huggingface_hub=={hf_ver}"], check=True)

    # The Kaggle kernel already has the *preinstalled* (older) huggingface_hub loaded in sys.modules
    # before this cell runs. The upgrade only changes the files on disk, so newly imported submodules
    # (e.g. _tree_cache.py from the new version) get mixed with old ones still in memory ->
    # "cannot import name 'as_extended_path' from huggingface_hub.utils._paths". Drop the stale
    # modules so everything below is re-imported from the freshly installed files.
    if any(m.startswith("datasets") for m in preloaded):
        # datasets registers pyarrow extension types at import; a re-import after the purge would
        # try to register them again and fail, so drop them from pyarrow's registry first.
        import pyarrow as pa
        for d in (2, 3, 4, 5):
            try: pa.unregister_extension_type(f"datasets.features.features.Array{d}DExtensionType")
            except Exception: pass
    for m in preloaded:
        sys.modules.pop(m, None)
    import huggingface_hub._snapshot_download   # noqa: F401  -- the import that used to fail

import torch, vllm
print("torch", torch.__version__, "| vllm", vllm.__version__, "| huggingface_hub", md.version("huggingface_hub"),
      "| GPUs", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


already imported before install: none
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/729.6 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.2.0 which is incompatible.
google-cloud-bigtable 2.36.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
pylibcudf-cu12 26.2.1 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.4.1 which is incompatible.
numba-cuda 0.22.2 requires cuda-core<1.0.0,>=0.3.2, but

torch 2.13.0+cu130 | vllm 0.30.0 | huggingface_hub 1.32.0 | GPUs 2 ['Tesla T4', 'Tesla T4']


In [4]:
# ============================ DATA ============================
import os, json, random, shutil, time
from collections import defaultdict
from datasets import load_dataset

os.makedirs(OUT_DIR, exist_ok=True)
ds = load_dataset("HuggingFaceH4/MATH-500", split="test")
pool = sorted([r for r in ds if int(r["level"]) == LEVEL], key=lambda r: r["unique_id"])
n = min(N_PER_LEVEL, len(pool))
rows = [pool[i] for i in sorted(random.Random(SEED * 100 + LEVEL).sample(range(len(pool)), n))]  # same subset for every model/mode
print(f"level {LEVEL}: {n}/{len(pool)} problems, K={K} -> {n*K} sequences @ {MAX_L} tokens")

with open(f"{OUT_DIR}/level{LEVEL}_problems.json", "w") as f:
    json.dump([r["unique_id"] for r in rows], f, indent=1)

README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/447k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

level 3: 43/105 problems, K=16 -> 688 sequences @ 8000 tokens


In [5]:
# ============================ MODEL ============================
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

import gc

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
THINK_END_ID = tok.convert_tokens_to_ids("</think>")

def gpu_free_frac():
    return [torch.cuda.mem_get_info(i)[0] / torch.cuda.mem_get_info(i)[1] for i in range(torch.cuda.device_count())]

if "llm" in globals():
    # Re-running this cell in a live kernel: the previous engine (and its tensor-parallel worker
    # processes) still hold both GPUs, so a second LLM() dies with "Free memory on device cuda:1
    # (0.4/14.56 GiB) on startup is less than desired GPU memory utilization". Shut it down first.
    print("shutting down previous vLLM engine...")
    try: llm.llm_engine.engine_core.shutdown()
    except Exception: pass
    del llm; gc.collect(); torch.cuda.empty_cache()
    for _ in range(60):                             # worker processes release memory asynchronously
        if min(gpu_free_frac()) >= GPU_MEM_UTIL: break
        time.sleep(1)

free = gpu_free_frac()
if min(free) < GPU_MEM_UTIL:
    raise RuntimeError(f"GPU free fraction {[round(x, 2) for x in free]} < GPU_MEM_UTIL={GPU_MEM_UTIL}: another process "
                       "holds the GPU. Restart the kernel (Run -> Restart) or lower GPU_MEM_UTIL.")

llm = LLM(
    model=MODEL_NAME,
    dtype="half",                                   # T4 has no bf16
    tensor_parallel_size=max(1, torch.cuda.device_count()),
    max_model_len=MAX_L + 512,                      # prompt (~200) + 8000 + forcing
    gpu_memory_utilization=GPU_MEM_UTIL,
    enable_prefix_caching=True,                     # forcing prompts share prefixes across L
    enforce_eager=ENFORCE_EAGER,
    seed=SEED,
)

# Qwen3 recommended sampling settings per mode
SP_GEN = SamplingParams(
    n=K, max_tokens=MAX_L, top_k=20,
    temperature=0.6 if THINK else 0.7,
    top_p=0.95 if THINK else 0.8,
)
SP_FORCE = SamplingParams(n=1, max_tokens=FORCE_TOKENS, temperature=0.0)

USER_SUFFIX = "\n\nPlease reason step by step, and put your final answer within \\boxed{}."
def prompt_ids(problem: str):
    msgs = [{"role": "user", "content": problem + USER_SUFFIX}]
    # Newer transformers return a BatchEncoding dict from apply_chat_template(tokenize=True);
    # list() of that gives the *keys* ("input_ids", "attention_mask"), not token ids.
    # Render to text and tokenize explicitly so this is a flat list[int] on every version.
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=THINK)
    return tok.encode(text, add_special_tokens=False)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

(Worker_TP1 pid=269) INFO 09-22 21:00:09 [model_runner.py:1066] Graph capturing finished in 15 secs, took 0.90 GiB
(Worker_TP0 pid=268) INFO 09-22 21:00:09 [model_runner.py:1066] Graph capturing finished in 15 secs, took 0.92 GiB
(Worker_TP1 pid=269) INFO 09-22 21:00:10 [gpu_worker.py:655] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9200 is equivalent to --gpu-memory-utilization=0.8547 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9853. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(Worker_TP0 pid=268) INFO 09-22 21:00:10 [gpu_worker.py:640] Available KV cache memory: 8.1 GiB
(Worker_TP0 pid=268) INFO 09-22 21:00:10 [gpu_worker.py:655] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9200 is equivalent to --gpu-memory-utilization=0.8539 without CUDA graph memory profiling. To 

Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:04<00:00,  7.66it/s]


(Worker_TP0 pid=268) INFO 09-22 21:00:55 [model_runner.py:1066] Graph capturing finished in 14 secs, took 0.43 GiB
(Worker_TP0 pid=268) INFO 09-22 21:00:55 [gpu_worker.py:825] CUDA graph pool memory: 0.43 GiB (actual), 0.96 GiB (estimated), difference: 0.53 GiB (124.1%).
(Worker_TP0 pid=268) INFO 09-22 21:00:55 [gpu_worker.py:888] Free memory on device (14.26/14.56 GiB) on startup. Desired GPU memory utilization is (0.92, 13.4 GiB). Actual usage is 4.0 GiB for consumed memory (weights + non-torch), 1.29 GiB for peak activation, and 0.43 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=8076915774` (7.52 GiB) to fit into requested memory, or `--kv-cache-memory=9008114176` (8.39 GiB) to fully utilize gpu memory. Current kv cache memory in use is 8.1 GiB.
(Worker_TP1 pid=269) INFO 09-22 21:00:55 [model_runner.py:1066] Graph capturing finished in 14 secs, took 0.43 GiB
(Worker_TP1 pid=269) INFO 09-22 21:00:55 [gpu_worker.py:825] CUDA graph pool memory:

(EngineCore pid=236) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Parse safetensors files: 100%|██████████| 3/3 [00:00<00:00,  8.25it/s]


(EngineCore pid=236) INFO 09-22 21:01:00 [kernel.py:416] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
INFO 09-22 21:01:02 [hf.py:642] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


In [6]:
# ============================ SCORING HELPERS ============================
import re
from math_verify import parse, verify

def extract_boxed(text):
    # content of the LAST \boxed{...} with balanced braces; None if absent
    i = text.rfind("\\boxed{")
    if i < 0: return None
    j, depth = i + len("\\boxed{"), 1
    while j < len(text) and depth:
        depth += {"{": 1, "}": -1}.get(text[j], 0); j += 1
    return text[i + 7 : j - 1].strip() if depth == 0 else text[i + 7 :].strip() or None

def safe_parse(s):
    if s is None: return []
    try:
        p = parse("\\boxed{" + s + "}", parsing_timeout=3)
        return p if p else parse("$" + s + "$", parsing_timeout=3)
    except Exception:
        return []

def safe_verify(a, b):
    try: return bool(a) and bool(b) and verify(a, b, timeout_seconds=3)
    except Exception: return False

def norm(s): return re.sub(r"\s+", "", s).replace("\\left","").replace("\\right","").replace("dfrac","frac")

def cluster(answers):
    # equivalence-class ids over a problem's answer pool (math_verify), -1 for no answer
    reps, cache, ids = [], {}, []
    for a in answers:
        if a is None: ids.append(-1); continue
        key = norm(a)
        if key in cache: ids.append(cache[key]); continue
        p, cid = safe_parse(a), None
        for rp, rid in reps:
            if safe_verify(rp, p): cid = rid; break
        if cid is None: cid = len(reps); reps.append((p, cid))
        cache[key] = cid; ids.append(cid)
    return ids

def force_suffix(prefix_ids):
    need_close = THINK and (THINK_END_ID not in prefix_ids)
    return ("\n</think>\n\n" if need_close else "\n\n") + "The final answer is \\boxed{"

In [7]:
# ============================ RUN: this (model, mode, level) ============================
# Resume support: on start, drop any partially-written problem (crash mid-write) and skip
# problems that already have all K samples recorded. Each problem is generated, scored,
# appended, and pushed to GitHub before moving to the next -- so a session that dies
# partway through a level loses at most the one problem it was working on.
if os.path.exists(OUT_PATH):
    with open(OUT_PATH) as f:
        existing = [json.loads(line) for line in f]
    counts = defaultdict(int)
    for x in existing:
        counts[x["unique_id"]] += 1
    done_ids = {uid for uid, c in counts.items() if c >= K}
    kept = [x for x in existing if x["unique_id"] in done_ids]
    if len(kept) != len(existing):
        with open(OUT_PATH, "w") as f:
            for x in kept:
                f.write(json.dumps(x, ensure_ascii=False) + "\n")
else:
    done_ids = set()

todo = [r for r in rows if r["unique_id"] not in done_ids]
print(f"[level {LEVEL}] {len(done_ids)}/{len(rows)} problems already done (resuming) -> {len(todo)} left")

t0 = time.time()
tok_total = 0          # generated tokens so far this session (vLLM's own bars only show per-call rates)
for i, r in enumerate(todo, 1):
    pt0 = time.time()

    # ---- 1. generate K samples at the max length, once ----
    # flush=True: in a headless commit run this is the only way to see which problem is in flight
    # while it is still generating -- Kaggle's log tab buffers, and vLLM's bars go to stderr.
    print(f"[level {LEVEL}] {i}/{len(todo)} START {r['unique_id']} ({r['subject']}) "
          f"-- {K} samples x {MAX_L} tok", flush=True)
    pid = prompt_ids(r["problem"])
    out = llm.generate([{"prompt_token_ids": pid}], SP_GEN)[0]
    gen_tok = sum(len(o.token_ids) for o in out.outputs)
    n_capped = sum(o.finish_reason == "length" for o in out.outputs)
    print(f"[level {LEVEL}] {i}/{len(todo)} gen {gen_tok:,} tok "
          f"(avg {gen_tok // K:,}/sample, {gen_tok / max(1e-9, time.time() - pt0):.0f} tok/s, "
          f"{n_capped}/{K} hit the {MAX_L} cap)", flush=True)

    # ---- 2. truncate to each L; queue budget-forcing where truncated ----
    recs = {}
    force_reqs = []   # (j, L, prompt_token_ids)
    for j, o in enumerate(out.outputs):
        full_ids = list(o.token_ids)
        rec = {"unique_id": r["unique_id"], "level": LEVEL, "subject": r["subject"], "gold": r["answer"],
               "sample": j, "full_tokens": len(full_ids), "finish_reason": o.finish_reason,
               "text": o.text, "by_L": {}}
        for L in LENGTHS:
            natural = len(full_ids) <= L and o.finish_reason == "stop"
            if natural:
                ans_text = o.text.split("</think>")[-1] if THINK else o.text
                rec["by_L"][str(L)] = {"tokens": len(full_ids), "truncated": False, "forced": False,
                                       "answer": extract_boxed(ans_text)}
            else:
                pre = full_ids[:L]
                force_reqs.append((j, L, pid + pre + tok.encode(force_suffix(pre), add_special_tokens=False)))
                rec["by_L"][str(L)] = {"tokens": len(pre), "truncated": True, "forced": True, "answer": None}
        recs[j] = rec

    # ---- 3. budget forcing: one batched greedy pass (prefix cache shares work across L) ----
    force_tok = 0
    if force_reqs:
        force_reqs.sort(key=lambda x: (x[0], x[1]))
        print(f"[level {LEVEL}] {i}/{len(todo)} forcing {len(force_reqs)} truncated prefixes "
              f"x {FORCE_TOKENS} tok", flush=True)
        fouts = llm.generate([{"prompt_token_ids": q[2]} for q in force_reqs], SP_FORCE)
        force_tok = sum(len(fo.outputs[0].token_ids) for fo in fouts)
        for (j, L, _), fo in zip(force_reqs, fouts):
            cont = fo.outputs[0]
            d = recs[j]["by_L"][str(L)]
            d["tokens"] += len(cont.token_ids)
            d["forced_text"] = cont.text
            d["answer"] = extract_boxed("\\boxed{" + cont.text)

    # ---- 4. score vs gold + equivalence-class ids over this problem's whole pool ----
    gold = safe_parse(r["answer"])
    keys = [(j, L) for j in range(K) for L in LENGTHS]
    pool = [recs[j]["by_L"][str(L)]["answer"] for j, L in keys]
    cids = cluster(pool)
    cid_correct = {}
    for (j, L), a, c in zip(keys, pool, cids):
        d = recs[j]["by_L"][str(L)]
        if c >= 0 and c not in cid_correct:
            cid_correct[c] = safe_verify(gold, safe_parse(a))
        d["cluster"] = c
        d["correct"] = bool(cid_correct.get(c, False))

    # ---- 5. checkpoint this problem, then push so a crash never loses more than it ----
    with open(OUT_PATH, "a") as f:
        for j in range(K):
            f.write(json.dumps(recs[j], ensure_ascii=False) + "\n")
    done_ids.add(r["unique_id"])
    if len(done_ids) % PUSH_EVERY == 0 or len(done_ids) == len(rows):
        push_outputs(f"{MODEL_TAG}/{MODE}/level{LEVEL}: {len(done_ids)}/{len(rows)} problems")
    tok_total += gen_tok + force_tok
    el = time.time() - pt0
    eta = (time.time() - t0) / i * (len(todo) - i) / 60
    print(f"[level {LEVEL}] {len(done_ids)}/{len(rows)} DONE {r['unique_id']} in {el:.0f}s "
          f"| {gen_tok + force_tok:,} tok ({gen_tok:,} gen + {force_tok:,} forced) "
          f"| session total {tok_total:,} tok | ETA {eta:.0f} min", flush=True)

# ---- final summary over the whole (now complete) file ----
with open(OUT_PATH) as f:
    all_recs = [json.loads(line) for line in f]
n = len(all_recs)
acc   = {L: sum(x["by_L"][str(L)]["correct"]   for x in all_recs) / n for L in LENGTHS}
trunc = {L: sum(x["by_L"][str(L)]["truncated"] for x in all_recs) / n for L in LENGTHS}
print(f"[level {LEVEL}] DONE {(time.time()-t0)/60:.1f} min -> {OUT_PATH}\n  per-sample acc {acc}\n  truncation     {trunc}")

[level 3] 0/43 problems already done (resuming) -> 43 left
[level 3] 1/43 START test/algebra/1098.json (Algebra) -- 16 samples x 8000 tok


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(Worker_TP0 pid=268) WARNING 09-22 21:01:04 [jit_monitor.py:141] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.
(Worker_TP0 pid=268) WARNING 09-22 21:01:10 [jit_monitor.py:141] Triton kernel JIT compilation during inference: _topk_topp_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(Worker_TP0 pid=268) WARNING 09-22 21:01:10 [jit_monitor.py:141] Triton kernel JIT compilation during inference: _topp_sb_stats_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(Worker_TP0 pid=268) WARNING 09-22 21:01:11 [jit_monitor.py:141] Triton kernel JIT compilation during inference: _topp_sb_step_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(Worker_TP0 pid=268) WARNING 09-22 21:01:13 [jit_monitor.py:141] Triton kernel JIT compilation during inference: _topp_sb_mask

Processed prompts: 100%|██████████| 16/16 [02:22<00:00,  8.88s/it, est. speed input: 4.06 toks/s, output: 211.62 toks/s]

...

Processed prompts: 100%|██████████| 32/32 [00:01<00:00, 25.66it/s, est. speed input: 21656.36 toks/s, output: 127.64 toks/s]


[main c133a72] Qwen3-4B/think/level3: 43/43 problems
 1 file changed, 16 insertions(+)


From https://github.com/bhuwanadhikari/sequence-vs-parallel-cot-in-slm
 * branch            main       -> FETCH_HEAD


Current branch main is up to date.
[level 3] 43/43 DONE test/precalculus/819.json in 74s | 20,774 tok (20,615 gen + 159 forced) | session total 2,551,430 tok | ETA 0 min
[level 3] DONE 285.8 min -> /kaggle/working/repo/outputs/Qwen3-4B/think/level3.jsonl
  per-sample acc {8000: 0.9636627906976745, 4000: 0.9563953488372093, 2000: 0.8895348837209303, 1000: 0.7994186046511628, 500: 0.5552325581395349}
  truncation     {8000: 0.09593023255813954, 4000: 0.30959302325581395, 2000: 0.7616279069767442, 1000: 0.998546511627907, 500: 1.0}


To https://github.com/bhuwanadhikari/sequence-vs-parallel-cot-in-slm.git
   a9f6d42..c133a72  main -> main


## Quick iso-budget summary (reads every `level*.jsonl` under `OUT_ROOT` and `EXTRA_INPUTS`)

Per (model, mode, difficulty) and each iso-budget point `N×L`:
`maj@N` (Monte-Carlo over N-subsets of the 16 samples, votes over equivalence classes), `pass@N` (unbiased estimator),
`gap = pass@N − maj@N`, measured mean total tokens (`N × mean tokens at L`), truncation rate, and 95% problem-level bootstrap CIs.
Full analysis / plotting is deferred to the repo; this cell is just a sanity read-out.

In [8]:
import glob, numpy as np
from math import comb

DIFF = {1: "easy", 2: "easy", 3: "medium", 4: "hard", 5: "hard"}
GRID = list(zip([1, 2, 4, 8, 16], LENGTHS))       # (N, L) iso-budget diagonal
B, M = 1000, 200                                   # bootstrap resamples, MC subsets for maj@N
rs = np.random.default_rng(SEED)

def pass_at_n(c, k, n): return 1.0 - comb(k - c, n) / comb(k, n)

def maj_at_n(clusters, correct, n):
    # clusters/correct: arrays of length K for one (problem, L). MC over random N-subsets; ties broken at random.
    k = len(clusters); wins = 0
    for _ in range(M):
        idx = rs.choice(k, n, replace=False)
        cs = clusters[idx]; cs = cs[cs >= 0]
        if len(cs) == 0: continue
        vals, cnt = np.unique(cs, return_counts=True)
        top = vals[cnt == cnt.max()]; pick = rs.choice(top)
        wins += bool(correct[idx][clusters[idx] == pick][0])
    return wins / M

def ci(x):
    x = np.asarray(x); bs = [x[rs.integers(0, len(x), len(x))].mean() for _ in range(B)]
    return np.percentile(bs, [2.5, 97.5])

files = defaultdict(list)                          # (model, mode) -> level files from all roots
for root in [OUT_ROOT] + list(EXTRA_INPUTS):
    for fn in glob.glob(f"{root}/*/*/level*.jsonl"):
        model, mode = fn.split("/")[-3:-1]; files[(model, mode)].append(fn)

for (model, mode), fns in sorted(files.items()):
    probs = defaultdict(list)
    for fn in fns:
        for line in open(fn):
            r = json.loads(line); probs[r["unique_id"]].append(r)
    print(f"\n=== {model} / {mode} ({len(probs)} problems from {sorted(os.path.basename(f) for f in fns)}) ===")
    print(f"{'diff':7}{'N x L':9}{'maj@N':>7}{'[95% CI]':>16}{'pass@N':>8}{'gap':>7}{'tokens':>8}{'trunc':>7}")
    for diff in ["easy", "medium", "hard", "all"]:
        P = [v for v in probs.values() if diff == "all" or DIFF[v[0]["level"]] == diff]
        if not P: continue
        for N, L in GRID:
            mj, ps, tk, tr = [], [], [], []
            for samples in P:
                d = [s["by_L"][str(L)] for s in sorted(samples, key=lambda s: s["sample"])]
                cl = np.array([x["cluster"] for x in d]); co = np.array([x["correct"] for x in d])
                mj.append(maj_at_n(cl, co, N)); ps.append(pass_at_n(int(co.sum()), len(d), N))
                tk.append(np.mean([x["tokens"] for x in d]) * N); tr.append(np.mean([x["truncated"] for x in d]))
            lo, hi = ci(mj)
            print(f"{diff:7}{N:>2}x{L:<6}{np.mean(mj):7.3f}  [{lo:.3f}, {hi:.3f}] {np.mean(ps):8.3f}{np.mean(ps)-np.mean(mj):7.3f}{np.mean(tk):8.0f}{np.mean(tr):7.2f}")


=== Qwen3-4B / think (130 problems from ['level1.jsonl', 'level2.jsonl', 'level3.jsonl', 'level4.jsonl']) ===
diff   N x L      maj@N        [95% CI]  pass@N    gap  tokens  trunc
easy    1x8000    0.978  [0.951, 1.000]    0.979  0.001    2666   0.02
easy    2x4000    0.971  [0.935, 0.997]    0.979  0.008    4910   0.15
easy    4x2000    0.954  [0.911, 0.989]    0.971  0.017    7298   0.60
easy    8x1000    0.891  [0.826, 0.951]    0.930  0.039    8111   0.99
easy   16x500     0.766  [0.674, 0.855]    0.884  0.118    8238   1.00
medium  1x8000    0.963  [0.909, 1.000]    0.964  0.000    3658   0.10
medium  2x4000    0.956  [0.900, 0.994]    0.974  0.017    5834   0.31
medium  4x2000    0.915  [0.827, 0.977]    0.941  0.027    7676   0.76
medium  8x1000    0.831  [0.718, 0.928]    0.901  0.071    8129   1.00
medium 16x500     0.625  [0.488, 0.765]    0.767  0.142    8276   1.00
hard    1x8000    1.000  [1.000, 1.000]    1.000  0.000    4248   0.00
hard    2x4000    1.000  [1.000, 1.000

## After the run
`/kaggle/working/outputs/<model>/<mode>/level<LEVEL>.jsonl` is what you keep: **Save Version** (commit) or download it. Merge the 20 files (or mount them as datasets and list them in `EXTRA_INPUTS`) for the full analysis.
Record schema (one line per (problem, sample)): `unique_id, level, subject, gold, sample, full_tokens, finish_reason, text, by_L{L: {tokens, truncated, forced, answer, forced_text, cluster, correct}}`.